In [1]:
# variables = ['datadate', 'ticker', 'agr', 'bm', 'mom12m', 'mve', 'operprof', 'roeq', 'absacc', 'acc', 'aeavol', 'age', 'baspread', 'BETA', 'bm_ia', 
#              'cash', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chfeps', 'chinv', 'chmom', 'chnanalyst', 
#              'chpmia', 'chtx', 'cinvest', 'convind', 'currat', 'depr', 'disp', 'divi', 'divo', 'dy', 'ear', 'egr', 'ep', 'fgr5yr', 
#              'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'idiovol', 'ill', 'indmom', 'invest', 'IPO', 'lev', 'mom1m', 'mom36m', 'ms', 
#              'mve_ia', 'nanalyst', 'nincr', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchsale_pchinvt', 
#              'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'pricedelay', 'ps', 'rd', 'rd_mve', 'rd_sale', 'realestate', 
#              'retvol', 'roaq', 'roavol', 'roic', 'rsup', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sfe', 'sgr', 'sin', 
#              'sp', 'std_dolvol', 'std_turn', 'stdcf', 'sue', 'tang', 'tb', 'turn', 'zerotrade']
variables = ['datadate', 'ticker', 'bm', 'mom12m', 'mve']

In [2]:
# import pandas as pd
# import numpy as np

# green_cleaned = pd.read_csv('../green cleaned.csv')
# green_cleaned['ret_fwd_1'] = (green_cleaned.groupby('permno')['ret_excess'].shift(-1) )

In [3]:
# green_cleaned = green_cleaned[green_cleaned['datadate'] < '2020-01-31']
# green_cleaned.loc[green_cleaned['datadate'] == '2019-12-31', 'ret_fwd_1'] = np.nan

In [4]:
# green_cleaned.to_csv('green_cleaned_pre_2020.csv', index=False)

In [2]:
from dotenv import find_dotenv, load_dotenv
from getpass import getpass
import os

load_dotenv(find_dotenv(), override=True)
#os.environ["OPENAI_MODEL_ID"] = "openai/gpt-4o-2024-08-06"
# 1. Stop Jupyter from crashing due to CrewAI asking for input in the background
os.environ["CREWAI_TELEMETRY_OPT_OUT"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [7]:
import os
import sqlite3
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# ============================================
# 1. ENVIRONMENT & PYTORCH FIXES
# ============================================
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Inject missing rms_norm for older PyTorch versions
if not hasattr(F, 'rms_norm'):
    def custom_rms_norm(input_tensor, normalized_shape, weight=None, eps=1e-5):
        variance = input_tensor.pow(2).mean(-1, keepdim=True)
        output = input_tensor * torch.rsqrt(variance + eps)
        if weight is not None:
            output = output * weight
        return output
    F.rms_norm = custom_rms_norm

# ============================================
# 2. LOAD THE MODEL LOCALLY
# ============================================
model_path = "./Model"  # Change to your local downloaded folder if different
print(f"Loading {model_path} into memory...")

device = torch.device("cpu")

# We strictly use 2048 to respect the model's physical architecture
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, model_max_length=2048)
model = AutoModelForCausalLM.from_pretrained(
    model_path, trust_remote_code=True, torch_dtype="bfloat16", device_map=None  # <--- Forbids offloading to hard drive
).to(device)
pipe = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=800,  # Limits output to leave room for the prompt
    do_sample=False,     # Temperature = 0 (Greedy decoding)
    return_full_text=False
)
print("✅ Model loaded successfully!")

# ============================================
# 3. DATA EXTRACTION (Pre-LLM)
# ============================================
# Instead of letting the LLM struggle with tools, we fetch the context for it.
print("\nFetching context from database...")

def get_market_context(date='2020-12-31'):
    try:
        conn = sqlite3.connect('firm_data.db')
        
        # 1. Total Firms
        total = pd.read_sql_query(f"SELECT COUNT(*) FROM firms WHERE datadate = '{date}'", conn).iloc[0,0]
        
        # 2. Extreme Value Firms (Undervalued)
        value_firms = pd.read_sql_query(f"""
            SELECT mve, bm, mom12m FROM firms 
            WHERE datadate = '{date}' AND bm > 1.5 ORDER BY bm DESC LIMIT 3
        """, conn).to_string(index=False)
        
        # 3. Extreme Momentum Firms
        mom_firms = pd.read_sql_query(f"""
            SELECT mve, bm, mom12m FROM firms 
            WHERE datadate = '{date}' AND mom12m > 1.5 ORDER BY mom12m DESC LIMIT 3
        """, conn).to_string(index=False)
        
        # 4. Intersection Test (Value + Momentum)
        combo = pd.read_sql_query(f"""
            SELECT COUNT(*) FROM firms 
            WHERE datadate = '{date}' AND bm > 0.5 AND mom12m > 0.5
        """, conn).iloc[0,0]
        
        conn.close()
        return total, value_firms, mom_firms, combo
    except Exception as e:
        return 500, "Data error", "Data error", 50

total_firms, top_value, top_mom, combo_count = get_market_context('2020-12-31')

# ============================================
# 4. CONSTRUCT THE OPTIMIZED PROMPT
# ============================================
# This prompt is clean, strict, and uses fewer than 800 tokens.
prompt = f"""You are an expert quantitative strategist developing systematic trading rules.

DATA CONTEXT (December 2020):
- Total firms available: {total_firms}
- 'mve' = log firm size (z-score)
- 'bm' = log book-to-market (z-score, high = undervalued)
- 'mom12m' = 12-month momentum (z-score)
- All features are standardized: mean = 0, standard deviation = 1.

SAMPLE EXTREMES:
Top Value (High BM):
{top_value}

Top Momentum (High MOM12M):
{top_mom}

Intersection: {combo_count} firms have both bm > 0.5 and mom12m > 0.5.

YOUR TASK:
Develop explicit, deterministic BUY/HOLD/SELL rules for generating trading signals using only these z-scores. 
Rely on established economic intuition (e.g., value and momentum premiums). Be precise with your thresholds (e.g., > 0.75, < -1.1).

OUTPUT FORMAT REQUIRED:
===========================================
SYSTEMATIC TRADING RULES
===========================================
BUY RULE:
if [Your complex z-score condition using AND/OR]:
    signal = BUY

SELL RULE:
if [Your complex z-score condition using AND/OR]:
    signal = SELL

HOLD RULE:
else:
    signal = HOLD

Rationale:
- [Why you chose these specific thresholds and combinations]
==========================================="""

# ============================================
# 5. GENERATE THE STRATEGY
# ============================================
print("\n🚀 Generating strategy rules (This will take a few seconds)...\n")

# Run the prompt through the PiT model
output = pipe(prompt)[0]['generated_text']

print("=" * 70)
print("SYSTEMATIC STRATEGY OUTPUT")
print("=" * 70)
print(output.strip())
print("=" * 70)

# Save to file
with open('systematic_strategy_202012_pit4b.txt', 'w') as f:
    f.write(output.strip())

print("\n✓ Strategy saved to 'systematic_strategy_202012_pit4b.txt'")

Loading ./Model into memory...


Device set to use cpu


✅ Model loaded successfully!

Fetching context from database...

🚀 Generating strategy rules (This will take a few seconds)...

SYSTEMATIC STRATEGY OUTPUT
[Your complex z-score condition using AND/OR]:
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score condition
- Your z-score conditio